# Day 60：毕业设计 Day 2 — 核心混合 RAG 与可溯源引用主链路实战

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段 **Day 60（`v1.0.0-day60`）**！
> 今天我们将端到端演练 `CapstoneRAGEngine`：从企业知识切片索引、BM25 + 向量双路召回、RRF 倒数排序融合、`CrossReranker` 精排，到带 `[1][2]` 行内引用的受控生成与低置信度防幻觉拒答。

## 1. 环境初始化与导入 `project_iterations/day_60` 模块

In [1]:
import re
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

day_60_dir = repo_root / "project_iterations" / "day_60"
if str(day_60_dir) not in sys.path:
    sys.path.insert(0, str(day_60_dir))

from zhishu_agent import (
    CapstoneRAGEngine,
    KnowledgeChunk,
    BM25Index,
    VectorIndex,
    reciprocal_rank_fusion,
)
print("成功加载 Day 60 模块:", day_60_dir)

成功加载 Day 60 模块: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_60


## 2. 构建企业知识库切片并完成双路索引

In [2]:
corpus = [
    KnowledgeChunk(
        chunk_id="kb-fin-01",
        doc_id="doc-fin",
        title="差旅与报销管理规范",
        section="第3章 报销时效",
        source="finance_policy_2026.md",
        content="员工出差返回后需在 15 个工作日内通过智枢财务系统提交报销单，单笔超过 5000 元需部门总监与财务复核。",
    ),
    KnowledgeChunk(
        chunk_id="kb-sre-502",
        doc_id="doc-sre",
        title="生产故障排查手册",
        section="ERR_502 网关超时",
        source="sre_runbook_v3.md",
        content="当线上出现 ERR_502 错误码时，优先检查 vLLM 推理 Pod 的显存占用与上游 Nginx proxy_read_timeout 配置（建议 >= 120s）。",
    ),
    KnowledgeChunk(
        chunk_id="kb-sec-01",
        doc_id="doc-sec",
        title="大模型数据安全准则",
        section="PII 脱敏",
        source="security_compliance.md",
        content="所有进入大模型推理链路的身份证号、银行卡号与手机号必须经过网关脱敏，审计日志严禁落盘明文 PII。",
    ),
]

engine = CapstoneRAGEngine(min_confidence=0.22, max_citations=2)
indexed_count = engine.index_documents(corpus)
print(f"已索引切片数: {indexed_count}, 引擎状态: {engine.stats()}")

已索引切片数: 3, 引擎状态: {'indexed_chunks': 3, 'embedding_dim': 256, 'rrf_k': 60, 'min_confidence': 0.22, 'max_citations': 2}


## 3. 混合检索（BM25 + Vector + RRF + Rerank）各阶段得分观察

In [3]:
query_sre = "线上网关报错 ERR_502 怎么排查？"
reranked, trace = engine.retrieve(query_sre, rerank_top_k=3)
print("检索追踪:", trace)
for item in reranked:
    print(
        f"chunk={item.chunk.chunk_id} | bm25={item.bm25_score:.3f}(r={item.bm25_rank}) "
        f"| vec={item.vector_score:.3f}(r={item.vector_rank}) "
        f"| rrf={item.rrf_score:.4f} | rerank={item.rerank_score:.4f}"
    )

检索追踪: {'bm25_hits': 3, 'vector_hits': 3, 'rrf_candidates': 3, 'reranked_count': 3, 'top_rerank_score': 0.554}
chunk=kb-sre-502 | bm25=11.328(r=1) | vec=0.259(r=1) | rrf=0.0328 | rerank=0.5540
chunk=kb-sec-01 | bm25=1.382(r=3) | vec=0.077(r=2) | rrf=0.0320 | rerank=0.2373
chunk=kb-fin-01 | bm25=1.582(r=2) | vec=0.033(r=3) | rrf=0.0320 | rerank=0.1690


## 4. 行内可溯源引用生成与低置信度拒答验证

In [4]:
# 4.1 域内问题：生成带 [1][2] 引用的答案并校验引用一致性
resp_ok = engine.query("员工出差报销单需要在几个工作日内提交？")
print("=== 域内高置信度问答 ===")
print(resp_ok.answer)
inline_ids = {int(m) for m in re.findall(r"\[(\d+)\]", resp_ok.answer)}
citation_ids = {c.citation_id for c in resp_ok.citations}
assert not resp_ok.refused and inline_ids.issubset(citation_ids)
print("引用一致性校验通过:", inline_ids)

# 4.2 域外问题：触发低置信度防幻觉拒答
resp_refuse = engine.query("明天沪深300指数会涨多少？")
print("\n=== 域外低置信度拒答 ===")
print(f"refused={resp_refuse.refused}, confidence={resp_refuse.confidence:.4f}")
print("reason:", resp_refuse.refusal_reason)
assert resp_refuse.refused is True and resp_refuse.citations == []

=== 域内高置信度问答 ===
员工出差返回后需在 15 个工作日内通过智枢财务系统提交报销单，单笔超过 5000 元需部门总监与财务复核。 [1]

参考来源：[1] 差旅与报销管理规范（finance_policy_2026.md）
引用一致性校验通过: {1}

=== 域外低置信度拒答 ===
refused=True, confidence=0.1855
reason: 最高重排置信度 0.1855 低于阈值 0.2200，触发防幻觉拒答
